In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping 

from sklearn.metrics import classification_report, confusion_matrix

## **Data Representation**

In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
df.head()

In [ ]:
df.info()

## Exploratory Data Analysis (EDA)

### 1. Dataset Overview
- The dataset contains **2,000 rows** and **21 columns**.
- The target variable is `price_range`, which is a **categorical variable** with four classes:
  - `0`: Low cost
  - `1`: Medium cost
  - `2`: High cost
  - `3`: Very high cost

### 2. Target Distribution

In [ ]:
df['price_range'].value_counts().plot(kind='bar', title='Distribution of Price Ranges')

### 3. Feature Types
- **Numerical Features**: `battery_power`, `ram`, `px_height`, `px_width`, etc.
- **Binary Features**: `blue`, `dual_sim`, `four_g`, `three_g`, `touch_screen`, `wifi`

### 4. Correlation with Target

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = df.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr[['price_range']].sort_values(by='price_range', ascending=False), annot=True, cmap='coolwarm')
plt.title('Correlation with Price Range')
plt.show()

### 5. Outlier and Distribution Check

In [ ]:
df[['battery_power', 'ram', 'px_height', 'px_width']].hist(figsize=(12, 8), bins=20)

### 6. Feature Redundancy
- `px_height` and `px_width` are both components of screen resolution.
- `pc` (primary camera) and `fc` (front camera) may correlate with price, but possibly non-linearly.

### 7. Missing Values

In [ ]:
df.isnull().sum()

## Step 3: Prepare the Data
- Split features and target
- Normalize features
- One-hot encode the target

In [ ]:
X = df.drop('price_range', axis=1)
y = df['price_range']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

y_encoded = to_categorical(y)

## Step 4: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

## Step 5: Build the Neural Network

In [ ]:
model = Sequential([
    Dense(128, input_dim=X_train.shape[1], activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(4, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## Step 7: Train the Model

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.1, epochs=20, batch_size=32)

## Step 8: Evaluate the Model

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.2f}")

## Step 9: Predictions and Performance Report

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print("Classification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

## Optional: Training Curves Visualization

In [ ]:
# Accuracy Plot
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()